# Test: WalmartBasePreprocessor + WalmartTabularFeatureEngineer

This notebook verifies that the two shared preprocessing classes (`base.py`, `tabular.py`)
work correctly before being used in any model-specific notebook.

Leakage check confirms that `transform_future()` computes lag/rolling features
using only the history passed to `fit()`, never the validation/test split's own
`Weekly_Sales`. A manually computed lag value (from history alone) is compared against
the class output for the same row; they must match exactly.

Missing value check confirms the dept-level fallback fills all NaN in the
lag/rolling columns for both the train and validation splits.

Aggregate feature check confirms all expected MarkDown aggregate columns
(created in `base.py`) are present and have reasonable values.

Type encoding check confirms label encoding (`Type_encoded`) and one-hot
encoding (`Type_A/B/C`) are both correctly assigned per store type.

In [1]:
import pandas as pd
import numpy as np

DATA_DIR = "../data/raw"
train = pd.read_csv(f"{DATA_DIR}/train.csv")
stores = pd.read_csv(f"{DATA_DIR}/stores.csv")
features = pd.read_csv(f"{DATA_DIR}/features.csv")

## Load classes and run WalmartBasePreprocessor

In [ ]:
from pathlib import Path
import sys
import importlib

# make imports work whether the notebook is run from /notebooks or repo root
cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "src").exists() else cwd.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.features.base as base
import src.features.tabular as tabular

importlib.reload(base)
importlib.reload(tabular)

from src.features.base import WalmartBasePreprocessor
from src.features.tabular import WalmartTabularFeatureEngineer

base_preprocessor = WalmartBasePreprocessor()
base_preprocessor.fit(stores, features)

train_merged = base_preprocessor.transform(train)
print("Train shape:", train_merged.shape)

expected_new_cols = [
    "markdown_available_period", "markdown_missing_count",
    "total_markdown", "abs_total_markdown",
    "positive_markdown_sum", "negative_markdown_sum",
    "has_markdown_signal",
]
missing_cols = [c for c in expected_new_cols if c not in train_merged.columns]
print("Missing expected columns:", missing_cols)
assert not missing_cols, f"Aggregate columns not found: {missing_cols}"

print("\nMarkDown aggregate sample stats:")
display(train_merged[expected_new_cols].describe())

Train shape: (421570, 42)
Missing expected columns: []

MarkDown aggregate sample stats:


,markdown_available_period,markdown_missing_count,total_markdown,abs_total_markdown,positive_markdown_sum,negative_markdown_sum,has_markdown_signal
count,421570.000000,421570.000000,421570.000000,421570.000000,421570.000000,421570.000000,421570.000000
mean,0.359210,3.374128,6684.041435,6684.243915,6684.142675,-0.101240,0.359210
std,0.479769,2.215686,14750.941551,14750.975434,14750.957865,4.303235,0.479769
min,0.000000,0.000000,0.000000,0.000000,0.000000,-265.760000,0.000000
25%,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,1.000000,5.000000,8075.260000,8075.260000,8075.260000,0.000000,1.000000
max,1.000000,5.000000,160510.610000,160510.610000,160510.610000,0.000000,1.000000


## Run WalmartTabularFeatureEngineer and verify leakage safety

In [3]:
cutoff = train_merged["Date"].max() - pd.Timedelta(weeks=39)
train_part = train_merged[train_merged["Date"] <= cutoff]
valid_part = train_merged[train_merged["Date"] > cutoff]

tab_fe = WalmartTabularFeatureEngineer()
tab_fe.fit(train_part)

X_train = tab_fe.transform_train(train_part)
X_valid = tab_fe.transform_future(valid_part)

# Leakage check: manually compute lag_1 for one validation row, compare to class output
sample = valid_part.sort_values(["Store", "Dept", "Date"]).iloc[0]
store, dept, date = sample["Store"], sample["Dept"], sample["Date"]

manual_check = train_part[
    (train_part["Store"] == store) & (train_part["Dept"] == dept)
].sort_values("Date")["Weekly_Sales"].iloc[-1]

engineered_value = X_valid[
    (X_valid["Store"] == store) & (X_valid["Dept"] == dept) & (X_valid["Date"] == date)
]["Sales_lag_1"].values[0]

print("Manual (from history only):", manual_check)
print("Engineered lag_1:", engineered_value)
assert manual_check == engineered_value, "LEAKAGE"

# NaN check on lag/rolling columns
lag_roll_cols = ["Sales_lag_1", "Sales_lag_4", "Sales_lag_52",
                  "Sales_roll_mean_4", "Sales_roll_std_4", "Sales_roll_mean_12"]

print("\nNaN in X_train:", X_train[lag_roll_cols].isna().sum().sum())
print("NaN in X_valid:", X_valid[lag_roll_cols].isna().sum().sum())
assert X_train[lag_roll_cols].isna().sum().sum() == 0
assert X_valid[lag_roll_cols].isna().sum().sum() == 0

print("\nAll checks passed")

Manual (from history only): 18378.16
Engineered lag_1: 18378.16

NaN in X_train: 0
NaN in X_valid: 0

All checks passed


## Type encoding check

In [4]:
print(X_train[["Type", "Type_encoded", "Type_A", "Type_B", "Type_C"]].drop_duplicates())

       Type  Type_encoded  Type_A  Type_B  Type_C
0         A             0       1       0       0
14899     B             1       0       1       0
208192    C             2       0       0       1
